# Data Science: Causal Inference - Double Machine Learning

## Estimating Treatment Effects in High Dimensions

**Challenge**: Estimate causal effect τ = E[Y(1) - Y(0)] when p >> n  
**Solution**: Double Machine Learning (Chernozhukov et al., 2018)

### Key Innovation
Use **Neyman-orthogonal scores** to make treatment effect estimation robust to first-stage ML errors.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import LassoCV, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_predict
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

## Part 1: Theory & Problem Setup

### Data Generating Process
```
Y = τ*T + f₀(X) + εᵧ  [outcome depends on treatment + confounders]
T = g₀(X) + εₜ         [treatment is confounded by X]
```

**Challenge**: With p >> n, naive OLS biases τ̂

**Solution**: Residualize both Y and T to remove confounding
```
Ỹ = Y - E[Y|X]
T̃ = T - E[T|X]
τ̂ = (∑ T̃Ỹ) / (∑ T̃²)
```


In [ ]:
# Generate data with confounding
n = 500
p = 100
tau_true = 1.0  # True treatment effect

X = np.random.randn(n, p)

# Treatment is confounded by X
T = np.sum(X[:, :10], axis=1) + np.random.randn(n) * 0.5

# Outcome: treatment effect + confounding
Y = tau_true * T + np.sum(X[:, :10] ** 2, axis=1) + np.random.randn(n) * 0.5

print(f"Data: n={n}, p={p}, τ_true={tau_true}")

## Part 2: DML Implementation


In [ ]:
class DoubleMachineLearning:
    """Double ML for high-dimensional treatment effects."""
    
    def __init__(self, Y, T, X):
        self.Y = Y
        self.T = T
        self.X = X
        self.n = len(Y)
    
    def estimate(self, method='lasso'):
        """Estimate treatment effect using DML."""
        
        # Step 1: Residualize treatment
        if method == 'lasso':
            model_t = LassoCV(cv=5)
        else:
            model_t = RandomForestRegressor(n_estimators=50, random_state=42)
        
        T_pred = cross_val_predict(model_t, self.X, self.T, cv=5)
        T_resid = self.T - T_pred
        
        # Step 2: Residualize outcome
        if method == 'lasso':
            model_y = LassoCV(cv=5)
        else:
            model_y = RandomForestRegressor(n_estimators=50, random_state=42)
        
        Y_pred = cross_val_predict(model_y, self.X, self.Y, cv=5)
        Y_resid = self.Y - Y_pred
        
        # Step 3: Estimate treatment effect
        tau_hat = np.sum(T_resid * Y_resid) / np.sum(T_resid ** 2)
        
        # Confidence interval
        residuals = Y_resid - tau_hat * T_resid
        sigma2 = np.mean(residuals ** 2)
        se = np.sqrt(sigma2 / np.sum(T_resid ** 2))
        
        return {
            'tau': tau_hat,
            'se': se,
            'ci': [tau_hat - 1.96*se, tau_hat + 1.96*se],
            'T_resid': T_resid,
            'Y_resid': Y_resid
        }

# Estimate with different methods
dml_lasso = DoubleMachineLearning(Y, T, X).estimate(method='lasso')
dml_rf = DoubleMachineLearning(Y, T, X).estimate(method='rf')

print("\nDOUBLE MACHINE LEARNING RESULTS")
print("="*60)
print(f"True effect:           {tau_true:.4f}")
print(f"DML (LASSO):           {dml_lasso['tau']:.4f} ± {dml_lasso['se']:.4f}")
print(f"DML (Random Forest):   {dml_rf['tau']:.4f} ± {dml_rf['se']:.4f}")
print(f"LASSO CI:              [{dml_lasso['ci'][0]:.4f}, {dml_lasso['ci'][1]:.4f}]")

## Part 3: Comparison with Naive Methods


In [ ]:
# Naive OLS
from sklearn.linear_model import LinearRegression

model_ols = LinearRegression()
model_ols.fit(np.column_stack([T, X]), Y)
tau_ols = model_ols.coef_[0]

# Naive LASSO (high bias)
model_lasso_full = LassoCV(cv=5)
model_lasso_full.fit(np.column_stack([T, X]), Y)
tau_lasso = model_lasso_full.coef_[0]

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Comparison
ax = axes[0]
methods = ['True', 'OLS\n(naive)', 'LASSO\n(naive)', 'DML\n(LASSO)', 'DML\n(RF)']
estimates = [tau_true, tau_ols, tau_lasso, dml_lasso['tau'], dml_rf['tau']]
errors = [0, abs(tau_ols - tau_true), abs(tau_lasso - tau_true), 
          abs(dml_lasso['tau'] - tau_true), abs(dml_rf['tau'] - tau_true)]
colors = ['green', 'red', 'orange', 'blue', 'purple']

for i, (est, err, col) in enumerate(zip(estimates, errors, colors)):
    ax.scatter(i, est, s=200, color=col, zorder=3)
    ax.annotate(f'{est:.3f}\nerr={err:.3f}', xy=(i, est), 
                xytext=(0, 15), textcoords='offset points', ha='center')

ax.axhline(y=tau_true, color='green', linestyle='--', alpha=0.5)
ax.set_xticks(range(len(methods)))
ax.set_xticklabels(methods)
ax.set_ylabel('Treatment Effect Estimate')
ax.set_title('Method Comparison: High-Dimensional Confounding')
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim([tau_true - 0.5, tau_true + 0.5])

# Residual plot (DML LASSO)
ax = axes[1]
ax.scatter(dml_lasso['T_resid'], dml_lasso['Y_resid'], alpha=0.4, s=20)
z = np.polyfit(dml_lasso['T_resid'], dml_lasso['Y_resid'], 1)
p = np.poly1d(z)
x_line = np.linspace(dml_lasso['T_resid'].min(), dml_lasso['T_resid'].max(), 100)
ax.plot(x_line, p(x_line), 'r-', linewidth=2, label=f'τ̂ = {dml_lasso["tau"]:.3f}')
ax.set_xlabel('Residual Treatment')
ax.set_ylabel('Residual Outcome')
ax.set_title('DML: Partialed-Out Relationship')
ax.legend()
ax.grid(True, alpha=0.3)

# Confidence intervals
ax = axes[2]
y_pos = [0, 1, 2, 3]
labels = ['DML-LASSO', 'DML-RF', 'OLS', 'LASSO']
ests = [dml_lasso['tau'], dml_rf['tau'], tau_ols, tau_lasso]
ses = [dml_lasso['se'], dml_rf['se'], 0.05, 0.05]

for i, (est, se, label) in enumerate(zip(ests, ses, labels)):
    ax.errorbar(est, i, xerr=1.96*se, fmt='o', markersize=8, capsize=5)
    ax.annotate(f'{est:.3f}', xy=(est, i), xytext=(5, 0), textcoords='offset points')

ax.axvline(x=tau_true, color='green', linestyle='--', linewidth=2, label='Truth')
ax.set_yticks(y_pos)
ax.set_yticklabels(labels)
ax.set_xlabel('Treatment Effect Estimate')
ax.set_title('95% Confidence Intervals')
ax.legend()
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('SECTION_2_DATA_SCIENCE/dml_causal_inference.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nCOMPARISON SUMMARY")
print("="*60)
print(f"{'Method':<20} {'Estimate':>12} {'Bias':>12}")
print("-"*60)
print(f"{'True Effect':<20} {tau_true:>12.4f}")
print(f"{'OLS (naive)':<20} {tau_ols:>12.4f} {tau_ols-tau_true:>12.4f}")
print(f"{'LASSO (naive)':<20} {tau_lasso:>12.4f} {tau_lasso-tau_true:>12.4f}")
print(f"{'DML (LASSO)':<20} {dml_lasso["tau"]:>12.4f} {dml_lasso["tau"]-tau_true:>12.4f}")
print(f"{'DML (RF)':<20} {dml_rf["tau"]:>12.4f} {dml_rf["tau"]-tau_true:>12.4f}")

## Key Insights

1. **Naive methods fail in high dimensions**: OLS & LASSO are severely biased
2. **DML is robust**: Orthogonalization removes confounding direction
3. **Flexible nuisance models**: Can use any ML method (RF, NN, etc.)
4. **Valid inference**: Asymptotically normal with correct standard errors

### References
- Chernozhukov, V., et al. (2018). "Double Machine Learning for Treatment and Structural Parameters"
